# Single Image Haze Removal Using Dark Channel Prior: Results Notebook

This notebook is the reproducibility and results companion for the software implementation deliverable. It follows the project report framing, verifies the RESIDE checkpoints, loads the quantitative evaluation report, and visualizes qualitative examples for the implemented dehazing methods.

The project compares the original Dark Channel Prior baseline with three later learning-based successor methods:

- AOD-Net
- DCPDN
- Color-Constrained Dehazing

## 1. Project Framing

Outdoor images captured in hazy conditions often suffer from reduced visibility, lower contrast, and distorted colors. This can reduce human interpretability and also harm downstream computer vision systems.

**Problem formulation:** Evaluate how the original Dark Channel Prior method and three later learning-based dehazing methods perform on the selected dehazing problem, with emphasis on image quality and practical usefulness.

**Hypothesis:** Later learning-based dehazing methods will outperform the original Dark Channel Prior method in the selected application setting, both quantitatively and visually.

**Datasets:** The report describes RESIDE training data, including ITS and OTS, and uses RESIDE-SOTS as the main evaluation benchmark. Both SOTS-Outdoor and SOTS-Indoor results are included. O-HAZE is used as an additional realistic-haze evaluation dataset.

**Checkpoint split used here:** This notebook selects the OTS RESIDE checkpoints for the reproducible run. SOTS-Outdoor is the closest evaluation match to that checkpoint split, while SOTS-Indoor and O-HAZE show transfer to indoor synthetic haze and real-world haze.


## 2. Method Lineage

The project requirement asks for a CVPR Best Paper Award winner and later papers that improve on or build from it. This implementation uses the following lineage:

| Role | Method | Paper/Source Role in Project |
|---|---|---|
| Baseline / original influential method | Dark Channel Prior (DCP) | Classic single-image dehazing method and CVPR Best Paper Award winner |
| Successor 1 | AOD-Net | Lightweight learning-based dehazing network |
| Successor 2 | DCPDN | Densely connected pyramid dehazing network with adversarial training |
| Successor 3 | Color-Constrained Dehazing | Larger learning-based method using color consistency constraints |

DCP is evaluated without training. The learning-based methods are trained with RESIDE data; this notebook uses the OTS checkpoint split for the reported reproducible run.


## 3. Reproducibility Notes

The notebook assumes the dataset folders, source files, and checkpoints are present in the project directory. The main scripts are:

```bash
./venv/bin/python train.py --model aodnet --train_set ots --epochs 20 --batch_size 8 --image_size 256
./venv/bin/python train.py --model dcpdn --train_set ots --epochs 20 --batch_size 8 --image_size 256
./venv/bin/python train.py --model color --train_set ots --epochs 20 --batch_size 8 --image_size 256
./venv/bin/python show_results.py --train_set ots
./venv/bin/python evaluate.py --train_set ots
```

The expected OTS checkpoints are:

```text
checkpoints/aodnet_ots_best.pth
checkpoints/dcpdn_ots_best.pth
checkpoints/color_dehaze_ots_best.pth
```

The indoor/ITS demonstration uses the non-suffixed checkpoints from the earlier RESIDE auto/ITS training run:

```text
checkpoints/aodnet_best.pth
checkpoints/dcpdn_best.pth
checkpoints/color_dehaze_best.pth
```

The quantitative report is read from `outputs/results.txt`. Run `show_results.py --train_set ots` first if you want to regenerate it with the same OTS checkpoint split used for the main table.

## 4. Setup

### Install system dependencies (Linux)
sudo apt-get update && sudo apt-get install -y \
    libgl1 \
    libxcb1

### Install Python dependencies
pip install -r requirements.txt

In [15]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

from config import (
    OUTPUTS_DIR,
    CHECKPOINTS_DIR,
    ITS_TRAIN_HAZY,
    ITS_TRAIN_GT,
    OTS_TRAIN_HAZY,
    OTS_TRAIN_GT,
    SOTS_INDOOR_HAZY,
    SOTS_INDOOR_GT,
    SOTS_OUTDOOR_HAZY,
    SOTS_OUTDOOR_GT,
    OHAZE_HAZY,
    OHAZE_GT,
)
from datasets import RESIDETestDataset, OHAZEDataset
from methods.aodnet import AODNet
from methods.dcpdn import DCPDN
from methods.color_dehaze import ColorConstrainedDehaze

CHECKPOINT_SPLIT = 'ots'
INDOOR_CHECKPOINT_SPLIT = 'its'
IMAGE_SIZE = 256
BASE_DIR = Path.cwd()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Working directory:', BASE_DIR)
print('Device:', device)
print('Main checkpoint split:', CHECKPOINT_SPLIT)
print('Indoor checkpoint split:', INDOOR_CHECKPOINT_SPLIT)

Working directory: /home/coder/IKT452/Computer_vision_project
Device: cuda
Main checkpoint split: ots
Indoor checkpoint split: its


## 5. Dataset and Checkpoint Sanity Checks

This section checks that the RESIDE training folders, evaluation datasets, and OTS-trained checkpoints exist before any result visualization is attempted.


In [16]:
def count_images(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(1 for item in path.iterdir() if item.suffix.lower() in {'.jpg', '.jpeg', '.png'})


def best_checkpoint_path(model_name, split=CHECKPOINT_SPLIT):
    """Return the best checkpoint for a split.

    The existing indoor checkpoints were produced before split suffixes were added,
    so split='its' first checks for a newer *_its_best.pth file and then falls
    back to the original non-suffixed *_best.pth file.
    """
    candidates = {
        'ots': [f'{model_name}_ots_best.pth'],
        'its': [f'{model_name}_its_best.pth', f'{model_name}_best.pth'],
        'auto': [f'{model_name}_best.pth'],
    }.get(split, [f'{model_name}_{split}_best.pth'])
    for filename in candidates:
        path = Path(CHECKPOINTS_DIR) / filename
        if path.exists():
            return path
    return Path(CHECKPOINTS_DIR) / candidates[0]


dataset_checks = pd.DataFrame([
    {'Dataset': 'ITS train hazy', 'Path': str(ITS_TRAIN_HAZY), 'Images': count_images(ITS_TRAIN_HAZY)},
    {'Dataset': 'ITS train GT', 'Path': str(ITS_TRAIN_GT), 'Images': count_images(ITS_TRAIN_GT)},
    {'Dataset': 'OTS train hazy', 'Path': str(OTS_TRAIN_HAZY), 'Images': count_images(OTS_TRAIN_HAZY)},
    {'Dataset': 'OTS train GT', 'Path': str(OTS_TRAIN_GT), 'Images': count_images(OTS_TRAIN_GT)},
    {'Dataset': 'SOTS-Indoor hazy', 'Path': str(SOTS_INDOOR_HAZY), 'Images': count_images(SOTS_INDOOR_HAZY)},
    {'Dataset': 'SOTS-Indoor GT', 'Path': str(SOTS_INDOOR_GT), 'Images': count_images(SOTS_INDOOR_GT)},
    {'Dataset': 'SOTS-Outdoor hazy', 'Path': str(SOTS_OUTDOOR_HAZY), 'Images': count_images(SOTS_OUTDOOR_HAZY)},
    {'Dataset': 'SOTS-Outdoor GT', 'Path': str(SOTS_OUTDOOR_GT), 'Images': count_images(SOTS_OUTDOOR_GT)},
    {'Dataset': 'O-HAZE hazy', 'Path': str(OHAZE_HAZY), 'Images': count_images(OHAZE_HAZY)},
    {'Dataset': 'O-HAZE GT', 'Path': str(OHAZE_GT), 'Images': count_images(OHAZE_GT)},
])

dataset_checks

,Dataset,Path,Images
0,ITS train hazy,/home/coder/IKT452/Computer_vision_project/Dat...,0
1,ITS train GT,/home/coder/IKT452/Computer_vision_project/Dat...,0
2,OTS train hazy,/home/coder/IKT452/Computer_vision_project/Dat...,0
3,OTS train GT,/home/coder/IKT452/Computer_vision_project/Dat...,0
4,SOTS-Indoor hazy,/home/coder/IKT452/Computer_vision_project/Dat...,500
5,SOTS-Indoor GT,/home/coder/IKT452/Computer_vision_project/Dat...,50
6,SOTS-Outdoor hazy,/home/coder/IKT452/Computer_vision_project/Dat...,500
7,SOTS-Outdoor GT,/home/coder/IKT452/Computer_vision_project/Dat...,492
8,O-HAZE hazy,/home/coder/IKT452/Computer_vision_project/Dat...,45
9,O-HAZE GT,/home/coder/IKT452/Computer_vision_project/Dat...,0


In [17]:
checkpoint_rows = []
for split_label, split in [('OTS', CHECKPOINT_SPLIT), ('ITS/auto', INDOOR_CHECKPOINT_SPLIT)]:
    for label, model_name in [
        ('AOD-Net', 'aodnet'),
        ('DCPDN', 'dcpdn'),
        ('Color-Constrained', 'color_dehaze'),
    ]:
        ckpt_path = best_checkpoint_path(model_name, split)
        checkpoint_rows.append({
            'Split': split_label,
            'Method': label,
            'Checkpoint': str(ckpt_path),
            'Exists': ckpt_path.exists(),
            'Size MB': round(ckpt_path.stat().st_size / (1024 * 1024), 2) if ckpt_path.exists() else None,
        })

checkpoint_df = pd.DataFrame(checkpoint_rows)
checkpoint_df

,Split,Method,Checkpoint,Exists,Size MB
0,OTS,AOD-Net,/home/coder/IKT452/Computer_vision_project/che...,True,0.02
1,OTS,DCPDN,/home/coder/IKT452/Computer_vision_project/che...,True,16.85
2,OTS,Color-Constrained,/home/coder/IKT452/Computer_vision_project/che...,True,89.45
3,ITS/auto,AOD-Net,/home/coder/IKT452/Computer_vision_project/che...,True,0.02
4,ITS/auto,DCPDN,/home/coder/IKT452/Computer_vision_project/che...,True,16.85
5,ITS/auto,Color-Constrained,/home/coder/IKT452/Computer_vision_project/che...,True,89.45


In [18]:
assert checkpoint_df['Exists'].all(), 'One or more expected OTS or ITS/auto checkpoints are missing.'
print('All expected OTS and ITS/auto checkpoints are present.')

All expected OTS and ITS/auto checkpoints are present.


## 6. Optional: Regenerate the Quantitative Report

Set `REGENERATE_RESULTS = True` if you want this notebook to rerun `show_results.py` and recreate `outputs/results.txt` using the OTS checkpoints. Leaving it as `False` keeps notebook execution faster and uses the already generated report.


In [19]:
REGENERATE_RESULTS = False

if REGENERATE_RESULTS:
    command = [sys.executable, 'show_results.py', '--train_set', CHECKPOINT_SPLIT]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Using existing outputs/results.txt. Set REGENERATE_RESULTS=True to recompute.')


Using existing outputs/results.txt. Set REGENERATE_RESULTS=True to recompute.


## 7. Load the Saved Text Report


In [20]:
results_path = Path(OUTPUTS_DIR) / 'results.txt'
if not results_path.exists():
    raise FileNotFoundError(f'Missing results file: {results_path}. Run show_results.py first.')

report_text = results_path.read_text()
print(report_text)


FileNotFoundError: Missing results file: /home/coder/IKT452/Computer_vision_project/outputs/results.txt. Run show_results.py first.

## 8. Parse the Summary Table


In [ ]:
summary_rows = []
in_summary = False
for line in report_text.splitlines():
    if line.strip().startswith('Method') and 'Num Images' in line:
        in_summary = True
        continue
    if in_summary and set(line.strip()) == {'-'}:
        continue
    if in_summary and not line.strip():
        break
    if not in_summary:
        continue

    parts = line.split()
    if len(parts) != 5:
        continue
    if parts[0] not in {'DCP', 'AOD-Net', 'DCPDN', 'Color-Constrained'}:
        continue

    summary_rows.append({
        'Method': parts[0],
        'Dataset': parts[1],
        'PSNR (dB)': float(parts[2]),
        'SSIM': float(parts[3]),
        'Num Images': int(parts[4]),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


## 9. Quantitative Result Plots

The report uses RESIDE-SOTS as the main benchmark and O-HAZE as an additional realistic-haze benchmark. Within SOTS, SOTS-Outdoor is the closest match to the OTS-trained checkpoints, while SOTS-Indoor helps show how the trained models transfer outside the outdoor training domain.

High PSNR = the model reconstructed the clean image more accurately

Low PSNR = the output differs more from the ground truth.

High SSIM = the dehazed image preserves image structure better.

Low SSIM = the dehazed image may have lost details, contrast, or natural structure.

In [ ]:
if summary_df.empty:
    raise RuntimeError('No parsed rows found in outputs/results.txt')

method_order = ['DCP', 'AOD-Net', 'DCPDN', 'Color-Constrained']
dataset_order = [name for name in ['SOTS-Outdoor', 'SOTS-Indoor', 'O-HAZE'] if name in set(summary_df['Dataset'])]
plot_df = summary_df.copy()
plot_df['Method'] = pd.Categorical(plot_df['Method'], categories=method_order, ordered=True)
plot_df['Dataset'] = pd.Categorical(plot_df['Dataset'], categories=dataset_order, ordered=True)
plot_df = plot_df.sort_values(['Dataset', 'Method'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, metric in zip(axes, ['PSNR (dB)', 'SSIM']):
    pivot = plot_df.pivot(index='Dataset', columns='Method', values=metric)
    pivot[method_order].plot(kind='bar', ax=ax)
    ax.set_title(metric)
    ax.set_xlabel('Dataset')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(title='Method', loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0)
plt.tight_layout(rect=(0, 0, 0.86, 1))
plt.show()


In [ ]:
primary = summary_df[summary_df['Dataset'] == 'SOTS-Outdoor'].sort_values('PSNR (dB)', ascending=False)
primary


## 10. Load the OTS-Trained Models

The following cell loads the OTS-trained successor models for qualitative inspection. DCP is not shown here because it is a classical algorithm without a learned checkpoint; it is included in the quantitative report.


In [ ]:
def load_checkpoint(model, model_name, device, split=CHECKPOINT_SPLIT):
    ckpt_path = best_checkpoint_path(model_name, split)
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Missing checkpoint: {ckpt_path}')
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()
    print(f'Loaded {model_name} ({split}): {ckpt_path}')
    return model


def load_models(device, split=CHECKPOINT_SPLIT):
    loaded_models = {}
    loaded_models['AOD-Net'] = load_checkpoint(AODNet().to(device), 'aodnet', device, split)
    loaded_models['DCPDN'] = load_checkpoint(DCPDN().to(device), 'dcpdn', device, split)
    loaded_models['Color-Constrained'] = load_checkpoint(
        ColorConstrainedDehaze().to(device), 'color_dehaze', device, split)
    return loaded_models

models = load_models(device, CHECKPOINT_SPLIT)
list(models.keys())

## 11. Build SOTS and O-HAZE Visual Sample Sets

Outdoor and indoor SOTS examples are both useful for the software deliverable. Outdoor examples are the closest visual match to the OTS-trained checkpoints, indoor examples demonstrate generalization within RESIDE-SOTS, and O-HAZE examples show realistic haze transfer.


In [ ]:
def make_dataset(name, hazy_dir, gt_dir, image_size=IMAGE_SIZE):
    hazy_dir = Path(hazy_dir)
    gt_dir = Path(gt_dir)
    if not hazy_dir.exists() or not gt_dir.exists():
        print(f'Skipping {name}: missing folder')
        return None
    dataset = RESIDETestDataset(hazy_dir, gt_dir, image_size=image_size)
    if len(dataset) == 0:
        print(f'Skipping {name}: no matched pairs')
        return None
    print(f'{name}: {len(dataset)} samples')
    return dataset


outdoor_dataset = make_dataset('SOTS-Outdoor', SOTS_OUTDOOR_HAZY, SOTS_OUTDOOR_GT)
indoor_dataset = make_dataset('SOTS-Indoor', SOTS_INDOOR_HAZY, SOTS_INDOOR_GT)


def make_ohaze_dataset(name, hazy_dir, gt_dir, image_size=IMAGE_SIZE):
    hazy_dir = Path(hazy_dir)
    gt_dir = Path(gt_dir)
    if not hazy_dir.exists() or not gt_dir.exists():
        print(f'Skipping {name}: missing folder')
        return None
    dataset = OHAZEDataset(hazy_dir, gt_dir, image_size=image_size)
    if len(dataset) == 0:
        print(f'Skipping {name}: no matched pairs')
        return None
    print(f'{name}: {len(dataset)} samples')
    return dataset


ohaze_dataset = make_ohaze_dataset('O-HAZE', OHAZE_HAZY, OHAZE_GT)


def valid_indices(dataset, requested=(0, 50, 120)):
    if dataset is None:
        return []
    return [idx for idx in requested if idx < len(dataset)]


outdoor_indices = valid_indices(outdoor_dataset)
indoor_indices = valid_indices(indoor_dataset)
ohaze_indices = valid_indices(ohaze_dataset, requested=(0, 15, 30))
print('Outdoor example indices:', outdoor_indices)
print('Indoor example indices:', indoor_indices)
print('O-HAZE example indices:', ohaze_indices)


## 12. Qualitative Visualization Helpers


In [ ]:
def run_model(model_name, model, hazy_batch):
    with torch.no_grad():
        if model_name == 'DCPDN':
            output, _, _ = model(hazy_batch)
        elif model_name == 'Color-Constrained':
            output, _ = model(hazy_batch)
        else:
            output = model(hazy_batch)
    return output


def plot_examples(dataset_name, dataset, example_indices, model_dict=None, split_label=CHECKPOINT_SPLIT):
    if model_dict is None:
        model_dict = models
    if dataset is None or not example_indices:
        print(f'No examples available for {dataset_name}')
        return

    num_rows = len(example_indices)
    num_cols = 2 + len(model_dict)
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(4 * num_cols, 4 * num_rows))
    if num_rows == 1:
        axes = [axes]

    for row_idx, sample_idx in enumerate(example_indices):
        hazy, gt, name = dataset[sample_idx]
        hazy_batch = hazy.unsqueeze(0).to(device)

        axes[row_idx][0].imshow(hazy.permute(1, 2, 0).cpu().numpy().clip(0, 1))
        axes[row_idx][0].set_title('Hazy' if row_idx == 0 else name)
        axes[row_idx][0].axis('off')

        axes[row_idx][1].imshow(gt.permute(1, 2, 0).cpu().numpy().clip(0, 1))
        axes[row_idx][1].set_title('Ground Truth' if row_idx == 0 else '')
        axes[row_idx][1].axis('off')

        for col_idx, (model_name, model) in enumerate(model_dict.items(), start=2):
            pred = run_model(model_name, model, hazy_batch)[0].detach().cpu()
            axes[row_idx][col_idx].imshow(pred.permute(1, 2, 0).numpy().clip(0, 1))
            axes[row_idx][col_idx].set_title(model_name if row_idx == 0 else '')
            axes[row_idx][col_idx].axis('off')

    fig.suptitle(f'{dataset_name} examples with {split_label.upper()} checkpoints', y=1.02)
    plt.tight_layout()
    plt.show()

## 13. Outdoor Qualitative Results


In [ ]:
plot_examples('SOTS-Outdoor', outdoor_dataset, outdoor_indices)


## 14. Indoor Generalization Examples


In [ ]:
plot_examples('SOTS-Indoor', indoor_dataset, indoor_indices)


## 15. Indoor Qualitative Results with ITS Checkpoints

This section uses the earlier indoor/ITS-style checkpoints to show that the trained learning-based methods can also dehaze indoor RESIDE images. These checkpoints are kept separate from the OTS checkpoints used for the main outdoor-focused result table.

In [ ]:
indoor_models = load_models(device, INDOOR_CHECKPOINT_SPLIT)
plot_examples(
    'SOTS-Indoor',
    indoor_dataset,
    indoor_indices,
    model_dict=indoor_models,
    split_label='ITS/auto',
)

## 16. O-HAZE Real-Haze Examples


In [ ]:
plot_examples('O-HAZE', ohaze_dataset, ohaze_indices)


## 17. Single-Sample Zoom View

Use this cell to inspect one image more closely. Change `ZOOM_DATASET` to `'outdoor'`, `'indoor'`, or `'ohaze'`.


In [ ]:
ZOOM_DATASET = 'outdoor'  # choose from 'outdoor', 'indoor', 'ohaze'
ZOOM_INDEX = 0
ZOOM_MODEL_SPLIT = 'ots'  # use 'its' for the indoor/ITS checkpoint view and 'ots' for the outdoor/OTS checkpoint view

zoom_options = {
    'outdoor': ('SOTS-Outdoor', outdoor_dataset, outdoor_indices),
    'indoor': ('SOTS-Indoor', indoor_dataset, indoor_indices),
    'ohaze': ('O-HAZE', ohaze_dataset, ohaze_indices),
}
if ZOOM_DATASET not in zoom_options:
    raise ValueError(f'Unknown ZOOM_DATASET: {ZOOM_DATASET}')
if ZOOM_MODEL_SPLIT not in {'ots', 'its'}:
    raise ValueError(f'Unknown ZOOM_MODEL_SPLIT: {ZOOM_MODEL_SPLIT}')
zoom_dataset_name, zoom_dataset, zoom_indices = zoom_options[ZOOM_DATASET]
zoom_models = indoor_models if ZOOM_MODEL_SPLIT == 'its' else models

if zoom_dataset is None or not zoom_indices:
    raise RuntimeError(f'No zoom examples available for {zoom_dataset_name}')

sample_idx = zoom_indices[min(ZOOM_INDEX, len(zoom_indices) - 1)]
hazy, gt, name = zoom_dataset[sample_idx]
hazy_batch = hazy.unsqueeze(0).to(device)

fig, axes = plt.subplots(1, 2 + len(zoom_models), figsize=(20, 4))
axes[0].imshow(hazy.permute(1, 2, 0).numpy().clip(0, 1))
axes[0].set_title(f'{zoom_dataset_name} hazy: {name}')
axes[0].axis('off')

axes[1].imshow(gt.permute(1, 2, 0).numpy().clip(0, 1))
axes[1].set_title('Ground Truth')
axes[1].axis('off')

for col_idx, (model_name, model) in enumerate(zoom_models.items(), start=2):
    pred = run_model(model_name, model, hazy_batch)[0].detach().cpu()
    axes[col_idx].imshow(pred.permute(1, 2, 0).numpy().clip(0, 1))
    axes[col_idx].set_title(model_name)
    axes[col_idx].axis('off')

fig.suptitle(f'{zoom_dataset_name} zoom with {ZOOM_MODEL_SPLIT.upper()} checkpoints', y=1.05)
plt.tight_layout()
plt.show()

## 18. Interpretation Summary

The expected interpretation is:

- RESIDE-SOTS is the main benchmark in the report; SOTS-Outdoor and SOTS-Indoor should both be discussed.
- OTS-trained learning-based models are expected to perform especially well on SOTS-Outdoor, which is closest to the outdoor training domain.
- The ITS/auto checkpoint section demonstrates that the same implemented learning-based methods can also dehaze indoor RESIDE images when trained on indoor RESIDE data.
- O-HAZE is a realistic-haze benchmark, so synthetic-to-real domain shift can reduce performance.
- If the later learning-based methods outperform DCP on the most relevant SOTS settings, this supports the hypothesis. Cases where DCP remains competitive should be discussed as limitations rather than hidden.